In [1]:
import numpy as np 
import faiss
import sys
import time
import csv
import os
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from numba import jit, prange
import threading
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm 
module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs, write_fvecs, read_ivecs
from src.utils import append_or_create_csv
from src.utils import compute_distance_tables_threaded, compute_distance_tables_vectorized, adc_distances_batch_numba\
                    , compute_all_distances_batch, adc_distances_all_optimized, lsq_alpha_from_codes \
                    , lsq_dot_tables_vectorized, lsq_distances_batch_numba
  
from datetime import datetime

# Current date & time
now = datetime.now()

# Format to a readable string
dt_str = now.strftime("%Y_%m_%d_%H_%M_%S")
print(dt_str)



2025_10_30_15_10_59


In [2]:
dataset_name = 'gist'

result_fp = f'/data/cpanourg/2-hdvc/results/vaq/{dataset_name}/{dt_str}'
os.makedirs(result_fp, exist_ok=True)

db_fp = '/data/cpanourg/2-hdvc/data/gist/gist_base.fvecs'
qr_fp = '/data/cpanourg/2-hdvc/data/gist/gist_query.fvecs'
gt_fp = '/data/cpanourg/2-hdvc/data/gist/gist_groundtruth.ivecs'
tr_fp = '/data/cpanourg/2-hdvc/temp/trainset.fvecs'
codes_fp = f'{result_fp}/codes.fvecs'
centroids_fp = f'{result_fp}/centroids.fvecs'

db = np.array(read_fvecs(db_fp))
qr = np.array(read_fvecs(qr_fp))

db_size = db.shape[0]
qr_size = qr.shape[0]
dim = db.shape[1]

train_ratio = 0.1
tr_size = int(train_ratio * db_size)

idxs = np.random.choice(db_size, size=tr_size, replace=False)
tr_set = db[idxs]

write_fvecs(tr_fp, tr_set)

Reading File - /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs:(1000000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs:(1000, 960)
Writing File - /data/cpanourg/2-hdvc/temp/trainset.fvecs:(100000, 960)


In [3]:
dataset_name = 'siftsmall'

result_fp = f'/data/cpanourg/2-hdvc/results/vaq/{dataset_name}/{dt_str}'
os.makedirs(result_fp, exist_ok=True)

db_fp = '/home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_base.fvecs'
qr_fp = '/home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_query.fvecs'
gt_fp = '/home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_groundtruth.ivecs'
tr_fp = '/home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/trainset.fvecs'
codes_fp = f'{result_fp}/codes.fvecs'
centroids_fp = f'{result_fp}/centroids.fvecs'

db = np.array(read_fvecs(db_fp))
qr = np.array(read_fvecs(qr_fp))

db_size = db.shape[0]
qr_size = qr.shape[0]
dim = db.shape[1]

train_ratio = 0.1
tr_size = int(train_ratio * db_size)

idxs = np.random.choice(db_size, size=tr_size, replace=False)
tr_set = db[idxs]

write_fvecs(tr_fp, tr_set)

Reading File - /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_base.fvecs:(10000, 128)
Reading File - /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_query.fvecs:(100, 128)
Writing File - /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/trainset.fvecs:(1000, 128)


In [4]:
topk = 100

In [ ]:
import subprocess
import os

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    f"{os.path.expanduser('~')}/local/glpk/lib:"
    f"{os.path.expanduser('~')}/local/armadillo/lib:"
    f"{env.get('CONDA_PREFIX', '')}/lib:"
    f"{env.get('LD_LIBRARY_PATH', '')}"
)

cmd = f"""
/home/cpanourg/projects/2-hdvc/lib/VAQ/build/examples/run_vaq \
  --dataset {db_fp} \
  --trainset {tr_fp} \
  --queries {qr_fp} \
  --file-format-ori fvecs \
  --timeseries-size {dim} \
  --dataset-size {db_size} \
  --trainset-size {tr_size} \
  --queries-size {qr_size} \
  --result {result_fp}/answer_vaq_VAQ256m32min7max8var1,HEAP_refine100,200_sift_10K.csv \
  --groundtruth {gt_fp} \
  --groundtruth-format ivecs \
  --method VAQ256m32min7max8var1,HEAP \
  --k {topk} \
  --refine 100,200 \
  --save-enc {codes_fp} \
  --save {centroids_fp}
"""

process = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Stream output line by line
for line in process.stdout:
    print(line, end="")
    if "Encoding time: " in line:
      enc_time_str = line
    if "Encoding time: " in line:
      enc_time_str = line
      

process.wait()
print(f"\n✅ Process finished with exit code {process.returncode}")


Arguments Passed:
	dataset = /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_base.fvecs
	dataset-size = 10000
	file-format-ori = fvecs
	groundtruth = /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_groundtruth.ivecs
	groundtruth-format = ivecs
	hc-bitalloc = 
	k = 100
	kmeans-ver = 0
	learn-ratio = 0.05
	method = VAQ256m32min7max8var1,HEAP
	queries = /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_query.fvecs
	queries-size = 100
	refine = 100,200
	result = /data/cpanourg/2-hdvc/results/vaq/siftsmall/2025_10_30_15_10_59/answer_vaq_VAQ256m32min7max8var1,HEAP_refine100,200_sift_10K.csv
	save = /data/cpanourg/2-hdvc/results/vaq/siftsmall/2025_10_30_15_10_59/centroids.fvecs
	save-enc = /data/cpanourg/2-hdvc/results/vaq/siftsmall/2025_10_30_15_10_59/codes.fvecs
	timeseries-size = 128
	trainset = /home/cpanourg/projects/2-hdvc/lib/VAQ/data/siftsmall/siftsmall_base.fvecs
	trainset-size = 10000
	visit-cluster = 1
Preprocessing steps..

Read datase

In [13]:
import numpy as np

with open(codes_fp, "rb") as f:
    nrows = np.fromfile(f, dtype=np.int64, count=1)[0]
    ncols = np.fromfile(f, dtype=np.int64, count=1)[0]
    codes = np.fromfile(f, dtype=np.int16, count=nrows * ncols).reshape(nrows, ncols)

print("Loaded codebook:", codes.shape, codes.dtype)



Loaded codebook: (1000000, 32) int16


In [14]:
codes

array([[  0,  86,  85, ..., 116,  86,  85],
       [  0, 116,   3, ...,  85, 127,  85],
       [  0, 127,  85, ..., 116,  86,  85],
       ...,
       [  0,  85,  85, ..., 116,  86, 116],
       [  0, 240,   3, ...,  84,  85, 116],
       [  0,  16, 104, ...,  85,   3, 144]], dtype=int16)